# AdaOS Node In Google Colab

Этот блокнот поднимает AdaOS-ноду в Google Colab и хранит состояние `.adaos` на Google Drive.

Что делает блокнот:
- монтирует Google Drive
- хранит `ADAOS_BASE_DIR` на Drive
- клонирует или обновляет репозиторий `adaos`
- запускает `tools/bootstrap.sh` с параметрами `--root-url`, `--zone`, `--join-code`, `--node-name`
- стартует локальный API в фоне
- дает быстрые команды для статуса, логов и бэкапа

Примечание: runtime Colab временный, но состояние в `.adaos` сохраняется на Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title Parameters
ROOT_URL = "https://ru.api.inimatic.com"  # @param {type:"string"}
ZONE = "ru"  # @param {type:"string"}
JOIN_CODE = ""  # @param {type:"string"}
NODE_NAME = "Colab Member"  # @param {type:"string"}
REV = "rev2026"  # @param {type:"string"}
REPO_URL = "https://github.com/inimatic/adaos.git"  # @param {type:"string"}
WORKSPACE_REGISTRY_REPO = "https://github.com/inimatic/adaos-registry.git"  # @param {type:"string"}

DRIVE_ROOT = "/content/drive/MyDrive/adaos_colab"  # @param {type:"string"}
ADAOS_DRIVE_DIR = f"{DRIVE_ROOT}/.adaos"  # @param {type:"string"}
REPO_DIR = "/content/adaos"  # @param {type:"string"}
LOG_DIR = f"{DRIVE_ROOT}/logs"  # @param {type:"string"}
PYTHON_VERSION = "3.11.9"  # @param {type:"string"}

SERVE_HOST = "127.0.0.1"  # @param {type:"string"}
SERVE_PORT = 8777  # @param {type:"integer"}
INSTALL_SERVICE = "never"  # @param ["never", "auto"]
NO_CORE_UPDATE = True  # @param {type:"boolean"}

FORCE_RECLONE = False  # @param {type:"boolean"}
FORCE_CLEAN_VENV = False  # @param {type:"boolean"}

print(
    {
        "ROOT_URL": ROOT_URL,
        "ZONE": ZONE,
        "JOIN_CODE_SET": bool(JOIN_CODE.strip()),
        "NODE_NAME": NODE_NAME,
        "REV": REV,
        "REPO_URL": REPO_URL,
        "WORKSPACE_REGISTRY_REPO": WORKSPACE_REGISTRY_REPO,
        "ADAOS_DRIVE_DIR": ADAOS_DRIVE_DIR,
        "REPO_DIR": REPO_DIR,
        "LOG_DIR": LOG_DIR,
        "PYTHON_VERSION": PYTHON_VERSION,
        "SERVE_HOST": SERVE_HOST,
        "SERVE_PORT": SERVE_PORT,
        "INSTALL_SERVICE": INSTALL_SERVICE,
        "NO_CORE_UPDATE": NO_CORE_UPDATE,
        "FORCE_RECLONE": FORCE_RECLONE,
        "FORCE_CLEAN_VENV": FORCE_CLEAN_VENV,
    }
)

In [ ]:
from pathlib import Path
import os
import shutil

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(ADAOS_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

if FORCE_RECLONE and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print("Drive root:", DRIVE_ROOT)
print("AdaOS base dir:", ADAOS_DRIVE_DIR)
print("Repo dir:", REPO_DIR)
print("Log dir:", LOG_DIR)

In [ ]:
import os
import shlex
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run(["git", "clone", "-b", REV, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REV], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if FORCE_CLEAN_VENV:
    venv_dir = repo / ".venv"
    if venv_dir.exists():
        subprocess.run(["rm", "-rf", str(venv_dir)], check=True)

print("Repo ready:", REPO_DIR)

In [ ]:
import os
import shlex
import subprocess
from datetime import datetime
from pathlib import Path

env = os.environ.copy()
env["ADAOS_BASE_DIR"] = ADAOS_DRIVE_DIR
env["PATH"] = f"/root/.local/bin:{env.get('PATH', '')}"

if "PYTHON_BIN" not in globals() or not str(PYTHON_BIN).strip():
    uv_bin = Path("/root/.local/bin/uv")
    if not uv_bin.exists():
        subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    subprocess.run(["bash", "-lc", f"uv python install {PYTHON_VERSION}"], env=env, check=True)
    py_find = subprocess.run(
        ["bash", "-lc", f"uv python find --managed-python {PYTHON_VERSION}"],
        env=env,
        check=True,
        capture_output=True,
        text=True,
    )
    PYTHON_BIN = py_find.stdout.strip()
    assert PYTHON_BIN, "uv did not return a Python path"

subprocess.run([PYTHON_BIN, "--version"], check=True)

cmd = [
    "bash",
    "tools/bootstrap_uv.sh",
    "--python",
    PYTHON_BIN,
    "--root-url",
    ROOT_URL,
    "--zone",
    ZONE,
    "--node-name",
    NODE_NAME,
    "--install-service",
    INSTALL_SERVICE,
    "--rev",
    REV,
]

if JOIN_CODE.strip():
    cmd += ["--join-code", JOIN_CODE.strip()]
if WORKSPACE_REGISTRY_REPO.strip():
    cmd += ["--workspace-registry-repo", WORKSPACE_REGISTRY_REPO.strip()]
if NO_CORE_UPDATE:
    cmd += ["--no-core-update"]

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%SZ")
bootstrap_log = Path(LOG_DIR) / f"bootstrap_{stamp}.log"
print("Running:", " ".join(shlex.quote(x) for x in cmd))
print("Log file:", bootstrap_log)

with open(bootstrap_log, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        fh.write(line)
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"bootstrap failed with exit code {rc}. See log: {bootstrap_log}")

print("Bootstrap finished successfully")

In [ ]:
import os
import subprocess
from pathlib import Path

uv_bin = Path("/root/.local/bin/uv")
if not uv_bin.exists():
    subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)

env = os.environ.copy()
env["PATH"] = f"/root/.local/bin:{env.get('PATH', '')}"

subprocess.run(["bash", "-lc", f"uv python install {PYTHON_VERSION}"], env=env, check=True)
py_find = subprocess.run(
    ["bash", "-lc", f"uv python find --managed-python {PYTHON_VERSION}"],
    env=env,
    check=True,
    capture_output=True,
    text=True,
)
PYTHON_BIN = py_find.stdout.strip()
assert PYTHON_BIN, "uv did not return a Python path"
subprocess.run([PYTHON_BIN, "--version"], check=True)
print("PYTHON_BIN =", PYTHON_BIN)

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path


def _adaos_env():
    env = os.environ.copy()
    env["ADAOS_BASE_DIR"] = ADAOS_DRIVE_DIR
    env["PYTHONUNBUFFERED"] = "1"
    return env


def _run_streaming(cmd, *, cwd=None, env=None, check=True):
    proc = subprocess.Popen(
        cmd,
        cwd=cwd or REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    stdout = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        stdout.append(line)
    output = "".join(stdout)
    result = subprocess.CompletedProcess(cmd, proc.wait(), stdout=output, stderr=None)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=output)
    return result


def _is_detached_api_command(args):
    tokens = [str(arg) for arg in args]
    return len(tokens) >= 2 and tokens[0] == "api" and tokens[1] in {"serve", "restart"}


def _run_adaos_detached(args):
    env = _adaos_env()
    logs_dir = Path(env["ADAOS_BASE_DIR"]) / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    log_path = logs_dir / "api-serve.log"
    pid_path = logs_dir / "api-serve.pid"
    quoted = " ".join(shlex.quote(str(arg)) for arg in args)
    cmd = ["bash", "-lc", "source .venv/bin/activate && exec python -u -m adaos " + quoted]
    log_file = open(log_path, "ab", buffering=0)
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdin=subprocess.DEVNULL,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_file.close()
    pid_path.write_text(str(proc.pid) + "\n", encoding="utf-8")
    message = f"Started detached AdaOS API command pid={proc.pid} log={log_path}"
    print(message, flush=True)
    return subprocess.CompletedProcess(cmd, 0, stdout=message + "\n", stderr=None)


def run_adaos(*args, check=True):
    if _is_detached_api_command(args):
        return _run_adaos_detached(args)
    env = _adaos_env()
    quoted = " ".join(shlex.quote(str(arg)) for arg in args)
    cmd = ["bash", "-lc", "source .venv/bin/activate && python -u -m adaos " + quoted]
    return _run_streaming(cmd, cwd=REPO_DIR, env=env, check=check)


def bash(script, check=True, cwd=None):
    return _run_streaming(["bash", "-lc", script], cwd=cwd or REPO_DIR, env=_adaos_env(), check=check)


def bootstrap_installed_skill_runtimes(skip_tests=True, check=True):
    args = ["--json"]
    if skip_tests:
        args.append("--skip-tests")
    quoted = " ".join(shlex.quote(arg) for arg in args)
    return bash(
        "source .venv/bin/activate\n"
        f"python -u -m adaos.apps.skill_runtime_migrate {quoted}",
        check=check,
    )


def wait_adaos_api(timeout=180, interval=2):
    url = f"http://{SERVE_HOST}:{SERVE_PORT}/api/node/status"
    deadline = time.monotonic() + float(timeout)
    last_error = None
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                body = response.read().decode("utf-8", errors="replace")
            print(body, flush=True)
            return subprocess.CompletedProcess(["wait_adaos_api", url], 0, stdout=body, stderr=None)
        except Exception as exc:
            last_error = exc
            time.sleep(float(interval))
    log_path = Path(ADAOS_DRIVE_DIR) / "logs" / "api-serve.log"
    print(f"AdaOS API did not become ready in {timeout}s: {last_error}", flush=True)
    if log_path.exists():
        print("Last api-serve.log lines:", flush=True)
        bash(f"tail -n 160 {shlex.quote(str(log_path))}", check=False)
    raise TimeoutError(f"AdaOS API did not become ready: {url}")


def start_adaos_api(timeout=180):
    try:
        return wait_adaos_api(timeout=1, interval=0.25)
    except Exception:
        pass
    run_adaos("api", "serve", "--host", SERVE_HOST, "--port", str(SERVE_PORT), check=False)
    return wait_adaos_api(timeout=timeout)


def install_skill_local(name):
    return run_adaos("skill", "install", name, "--local")


def install_scenario_dev(name):
    quoted_name = shlex.quote(str(name))
    return bash(
        "export ENV_TYPE=dev ADAOS_ENV_TYPE=dev\n"
        "source .venv/bin/activate\n"
        f"python -u -m adaos scenario install {quoted_name}"
    )


def switch_webspace_scenario(webspace, scenario_id, *, set_home=True, check=False):
    args = [
        "node",
        "yjs",
        "scenario",
        "--webspace",
        webspace,
        "--scenario-id",
        scenario_id,
        "--json",
    ]
    if set_home:
        args.append("--set-home")
    return run_adaos(*args, check=check)


def repair_adaos_workspace_git(check=True):
    script = r"""
set -u
repo="$ADAOS_BASE_DIR/workspace"
echo "Workspace: $repo"
if [ ! -d "$repo/.git" ]; then
  echo "No git repo at $repo"
  exit 0
fi

rm -f "$repo/.git/index.lock" "$repo/.git/shallow.lock" 2>/dev/null || true
if ! git -C "$repo" status --porcelain >/dev/null 2>&1; then
  echo "git status failed; rebuilding git index before repair."
  rm -f "$repo/.git/index" "$repo/.git/index.lock" 2>/dev/null || true
  git -C "$repo" reset --mixed HEAD || git -C "$repo" reset --hard HEAD || true
fi


echo "Status before repair:"
git -C "$repo" status --short | sed -n '1,80p' || true
dirty_count="$(git -C "$repo" status --porcelain | wc -l | tr -d ' ')"
echo "Dirty entries before repair: $dirty_count"
if [ "$dirty_count" = "0" ]; then
  exit 0
fi

staged_delete_count="$(git -C "$repo" diff --cached --name-only --diff-filter=D | wc -l | tr -d ' ')"
other_staged_count="$(git -C "$repo" status --porcelain | awk 'substr($0,1,1)!=" " && substr($0,1,2)!="??" && substr($0,1,1)!="D"{n++} END{print n+0}')"
if [ "$staged_delete_count" -ge 20 ] && [ "$other_staged_count" = "0" ]; then
  echo "Detected mass staged deletions ($staged_delete_count); restoring tracked files before stash/reset."
  git -C "$repo" diff --cached --name-only --diff-filter=D -z | xargs -0 -r git -C "$repo" restore --staged --worktree --
fi

dirty_count="$(git -C "$repo" status --porcelain | wc -l | tr -d ' ')"
if [ "$dirty_count" != "0" ]; then
  echo "Detected remaining local changes; creating safety stash before reset."
  git -C "$repo" stash push -u -m "adaos:manual-repair $(date -u +%Y%m%dT%H%M%SZ)" || true
fi

git -C "$repo" reset --hard HEAD || {
  echo "reset --hard failed; rebuilding git index and retrying."
  rm -f "$repo/.git/index"
  git -C "$repo" reset --hard HEAD
}

if [ -f "$repo/.git/info/sparse-checkout" ]; then
  tmp="$repo/.git/info/sparse-checkout.tmp"
  grep -v '^--' "$repo/.git/info/sparse-checkout" > "$tmp" || true
  mv "$tmp" "$repo/.git/info/sparse-checkout"
fi

git -C "$repo" sparse-checkout reapply || true
mid_count="$(git -C "$repo" status --porcelain | wc -l | tr -d ' ')"
if [ "$mid_count" != "0" ]; then
  echo "Workspace is still dirty after reset; disabling sparse-checkout and rebuilding the index."
  git -C "$repo" sparse-checkout disable || true
  rm -f "$repo/.git/index"
  git -C "$repo" reset --hard HEAD
fi


echo "Status after repair:"
git -C "$repo" status --short | sed -n '1,80p' || true
after_count="$(git -C "$repo" status --porcelain | wc -l | tr -d ' ')"
echo "Dirty entries after repair: $after_count"
test "$after_count" = "0"
"""
    return bash(script, check=check)


def refresh_adaos_workspace_registry(check=True):
    script = r"""
set -u
repo="$ADAOS_BASE_DIR/workspace"
echo "Workspace: $repo"
if [ ! -d "$repo/.git" ]; then
  echo "No git repo at $repo"
  exit 1
fi

rm -f "$repo/.git/index.lock" "$repo/.git/shallow.lock" 2>/dev/null || true
if ! git -C "$repo" status --porcelain >/dev/null 2>&1; then
  echo "git status failed; rebuilding git index before refresh."
  rm -f "$repo/.git/index" "$repo/.git/index.lock" 2>/dev/null || true
  git -C "$repo" reset --mixed HEAD || git -C "$repo" reset --hard HEAD || true
fi
echo "Remote:"
git -C "$repo" remote -v || true
echo "Before:"
git -C "$repo" log -1 --oneline --decorate || true

git -C "$repo" fetch --prune origin main
git -C "$repo" checkout -B main origin/main
git -C "$repo" reset --hard origin/main

if [ -f "$repo/.git/info/sparse-checkout" ]; then
  tmp="$repo/.git/info/sparse-checkout.tmp"
  grep -v '^--' "$repo/.git/info/sparse-checkout" > "$tmp" || true
  mv "$tmp" "$repo/.git/info/sparse-checkout"
fi

git -C "$repo" sparse-checkout disable || true
git -C "$repo" reset --hard origin/main

echo "After:"
git -C "$repo" log -1 --oneline --decorate || true
echo "Matching scenarios:"
git -C "$repo" ls-tree -d --name-only HEAD:scenarios 2>/dev/null | grep -Ei 'new|face|vision' || true
echo "Matching skills:"
git -C "$repo" ls-tree -d --name-only HEAD:skills 2>/dev/null | grep -Ei 'new|face|vision' || true
echo "Status:"
git -C "$repo" status --short | sed -n '1,80p' || true
"""
    return bash(script, check=check)


print("Helpers ready: run_adaos(...), bash(...), start_adaos_api(), wait_adaos_api(), bootstrap_installed_skill_runtimes(...), install_skill_local(...), install_scenario_dev(...), switch_webspace_scenario(...), repair_adaos_workspace_git(), refresh_adaos_workspace_registry()")

In [ ]:
# Reconcile persisted Drive skill runtimes with the current ephemeral Colab .venv.
# This is required after changing Colab runtime type, for example CPU -> T4.
bootstrap_installed_skill_runtimes(skip_tests=True)

In [ ]:
run_adaos("where")
run_adaos("node", "status", check=False)
bash(f"curl -sS http://{SERVE_HOST}:{SERVE_PORT}/api/node/status || true", check=False)

## Useful Operations

In [ ]:
# Start API in background and wait until /api/node/status is ready
# start_adaos_api(timeout=180)

# Detached start/restart also returns immediately from the cell
# run_adaos('api', 'serve', '--host', SERVE_HOST, '--port', str(SERVE_PORT), check=False)

# Stop API
# run_adaos('api', 'stop', check=False)

# Restart API in background
# run_adaos('api', 'restart', check=False)
# wait_adaos_api(timeout=180)

In [ ]:
# New Face Vision Colab flow
# repair_adaos_workspace_git()
# install_skill_local('new_face_vision_skill')
# install_scenario_dev('new_face_vision')
# bootstrap_installed_skill_runtimes(skip_tests=True)
# start_adaos_api(timeout=180)
# switch_webspace_scenario('desktop', 'new_face_vision')

In [ ]:
bash(f"curl -sS http://{SERVE_HOST}:{SERVE_PORT}/api/node/status || true", check=False)

In [ ]:
bash('find "$ADAOS_BASE_DIR" -maxdepth 2 -type f | sort | tail -n 80', check=False)

In [ ]:
from datetime import datetime
import os
import shlex

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%SZ")
snapshot = f"{LOG_DIR}/adaos_snapshot_{stamp}.tar.gz"
script = f"tar -czf {shlex.quote(snapshot)} -C {shlex.quote(DRIVE_ROOT)} .adaos"
bash(script)
print("Snapshot saved:", snapshot)

## Notes

- Если хочешь войти в существующую сеть как `member`, передай `JOIN_CODE`.
- Если `JOIN_CODE` пустой, bootstrap просто поднимет локальную ноду без `node join`.
- В Colab лучше оставлять `INSTALL_SERVICE="never"`, потому что systemd там обычно недоступен.
- Состояние ноды лежит в `ADAOS_DRIVE_DIR`, так что после переподключения Colab его можно переиспользовать.